# Phase 2 — Google Colab Execution Notebook

This notebook runs the tiered data pipeline in Google Colab.
- Repository: `https://github.com/mahendravelagapudi099-wq/Ultra-Data.git`
- Working directory: Project root (`/content/Ultra-Data` or `/content/drive/MyDrive/Ultra-Dataa`)
- Pipeline stages: L0 Substitute Generation → L1 Filtering → L2 Selection

In [ ]:
# Cell 1: Clone repository (recommended) or mount Google Drive
import os

repo_url = "https://github.com/mahendravelagapudi099-wq/Ultra-Data.git"
target_dir = "/content/Ultra-Data"

if not os.path.exists(target_dir):
    !git clone {repo_url} {target_dir}
else:
    print(f"{target_dir} already exists. Pulling latest updates...")
    !cd {target_dir} && git pull origin main

# Optional: Mount Google Drive if syncing via Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Cell 2: Change directory into the project root
import os

candidate_paths = [
    "/content/Ultra-Data",
    "/content/drive/MyDrive/Ultra-Data",
    "/content/drive/MyDrive/Ultra-Dataa",
]

for path in candidate_paths:
    if os.path.exists(path):
        os.chdir(path)
        print(f"Active project root: {path}")
        break
else:
    print(f"Current working directory: {os.getcwd()}")

!pwd
!ls -la

In [ ]:
# Cell 3: Install required packages if missing
!pip install -q -r requirements.txt

In [ ]:
# Cell 4: Set PYTHONPATH
import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.environ["PYTHONPATH"] = "."
print(f"Project root (CWD): {project_root}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH')}")

In [ ]:
# Cell 5: Generate expanded L0 substitute data
!PYTHONPATH=. python scripts/generate_l0_expanded.py

In [ ]:
# Cell 6: Run L1 expanded pipeline
!PYTHONPATH=. python scripts/run_l1.py --config configs/l1_expanded.yaml

In [ ]:
# Cell 7: Run L2 pipeline
!PYTHONPATH=. python scripts/run_l2.py --config configs/l2_tiny.yaml

In [ ]:
# Cell 8: Print output file locations
from pathlib import Path

print("=" * 60)
print("Pipeline Output File Locations & Status")
print("=" * 60)

data_dirs = [
    Path("data/l0_raw"),
    Path("data/l1_filtered"),
    Path("data/l2_selected"),
]

for directory in data_dirs:
    posix_path = directory.as_posix()
    print(f"\n[Directory] {posix_path}/")
    if directory.exists():
        files = sorted([f for f in directory.iterdir() if f.is_file()])
        if files:
            for f in files:
                size_kb = f.stat().st_size / 1024
                print(f"  -> {f.as_posix()} ({size_kb:.2f} KB)")
        else:
            print("  -> (empty)")
    else:
        print("  -> (directory does not exist yet)")

print("\n" + "=" * 60)